#### Final End-to-End Pipeline

This notebook integrates the four selected outputs:

1. Direction — pooled LightGBM classifier.
2. Magnitude — pooled LightGBM regressor.
3. Direction confidence — logistic correctness model.
4. Magnitude confidence — LightGBM expected-error model.

Model selection and hyperparameter tuning were completed using train and validation data.
The test split is scored once in this notebook and is not used for further model selection.

Final outputs:

- `predictions.csv`
- `actuals.csv`
- `statistics.csv`

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb

from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
)

warnings.filterwarnings("ignore")

SEED = 42
FINAL_DIRECTION_THRESHOLD = 0.64

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
FINAL_OUTPUT_DIR = PROJECT_ROOT / "final_outputs"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DATA_DIR)
print("Final output:", FINAL_OUTPUT_DIR)

Project root: /Users/kushagr/Desktop/astra-assignment
Processed data: /Users/kushagr/Desktop/astra-assignment/data/processed
Final output: /Users/kushagr/Desktop/astra-assignment/final_outputs


In [2]:
def resolve_single_file(
    preferred_path: Path,
    filename: str,
) -> Path:
    if preferred_path.exists():
        return preferred_path

    matches = list(
        PROJECT_ROOT.rglob(filename)
    )

    assert len(matches) == 1, (
        f"Expected exactly one {filename}, "
        f"found {len(matches)}: {matches}"
    )

    return matches[0]


MODEL_PANEL_PATH = resolve_single_file(
    PROCESSED_DATA_DIR / "magnitude_model_panel.parquet",
    "magnitude_model_panel.parquet",
)

DIRECTION_FEATURE_LIST_PATH = resolve_single_file(
    PROCESSED_DATA_DIR / "final_direction_feature_columns.json",
    "final_direction_feature_columns.json",
)

MAGNITUDE_CONFIG_PATH = resolve_single_file(
    PROCESSED_DATA_DIR / "final_magnitude_model_config.json",
    "final_magnitude_model_config.json",
)

print("Panel:", MODEL_PANEL_PATH)
print("Direction features:", DIRECTION_FEATURE_LIST_PATH)
print("Magnitude config:", MAGNITUDE_CONFIG_PATH)

Panel: /Users/kushagr/Desktop/astra-assignment/outputs/processed_data/magnitude_model_panel.parquet
Direction features: /Users/kushagr/Desktop/astra-assignment/outputs/processed_data/final_direction_feature_columns.json
Magnitude config: /Users/kushagr/Desktop/astra-assignment/data/processed/final_magnitude_model_config.json


In [3]:
model_df = pd.read_parquet(
    MODEL_PANEL_PATH
)

model_df["pred_date"] = pd.to_datetime(
    model_df["pred_date"]
)

if "target_date" in model_df.columns:
    model_df["target_date"] = pd.to_datetime(
        model_df["target_date"]
    )

model_df = (
    model_df
    .sort_values(["pred_date", "symbol"])
    .reset_index(drop=True)
)

with open(
    DIRECTION_FEATURE_LIST_PATH,
    "r",
    encoding="utf-8",
) as file:
    direction_feature_columns = json.load(file)

with open(
    MAGNITUDE_CONFIG_PATH,
    "r",
    encoding="utf-8",
) as file:
    magnitude_config = json.load(file)

print("Panel shape:", model_df.shape)
print("Symbols:", model_df["symbol"].nunique())
print("Direction features:", len(direction_feature_columns))
print(model_df["split"].value_counts(dropna=False))

Panel shape: (301275, 45)
Symbols: 208
Direction features: 26
split
train      186555
test        58656
valid       54009
embargo      2055
Name: count, dtype: int64


In [4]:
required_columns = [
    "symbol",
    "pred_date",
    "target_date",
    "split",
    "actual_return_pct",
    "actual_direction",
]

missing_required_columns = [
    column
    for column in required_columns
    if column not in model_df.columns
]

missing_direction_features = [
    column
    for column in direction_feature_columns
    if column not in model_df.columns
]

assert not missing_required_columns, (
    f"Missing required columns: {missing_required_columns}"
)

assert not missing_direction_features, (
    f"Missing direction features: {missing_direction_features}"
)

assert not model_df.duplicated(
    ["symbol", "pred_date"]
).any()

assert model_df["symbol"].nunique() == 208

assert model_df["split"].isin(
    ["train", "valid", "test", "embargo"]
).all()

assert model_df["actual_return_pct"].notna().all()

model_df["actual_direction"] = np.where(
    model_df["actual_return_pct"] >= 0,
    1,
    -1,
).astype("int8")

model_df["actual_magnitude_pct"] = (
    model_df["actual_return_pct"].abs()
)

print("Final panel validation passed.")

Final panel validation passed.


In [5]:
model_df["trailing_magnitude_20d"] = (
    model_df
    .groupby("symbol")["actual_magnitude_pct"]
    .transform(
        lambda series: (
            series.shift(1)
            .rolling(
                window=20,
                min_periods=20,
            )
            .mean()
        )
    )
)

assert (
    model_df["trailing_magnitude_20d"]
    .dropna()
    .ge(0)
    .all()
)

print(
    "Trailing baseline missing rows:",
    model_df["trailing_magnitude_20d"].isna().sum(),
)

Trailing baseline missing rows: 4160


In [6]:
explicitly_excluded_columns = {
    "pred_date",
    "target_date",
    "split",

    "actual_return_pct",
    "actual_direction",
    "actual_magnitude_pct",
    "log_actual_magnitude",
    "universe_mean_pct",

    "next_open",
    "next_open_price",
    "target_return",
    "target_direction",
    "target_magnitude",
}

suspicious_target_terms = [
    "actual_",
    "target_",
    "next_open",
    "future_",
    "forward_",
    "label",
]

magnitude_feature_columns = []

for column in model_df.columns:
    if column in explicitly_excluded_columns:
        continue

    if column == "symbol":
        magnitude_feature_columns.append(column)
        continue

    if any(
        term in column.lower()
        for term in suspicious_target_terms
    ):
        continue

    if pd.api.types.is_numeric_dtype(
        model_df[column]
    ):
        magnitude_feature_columns.append(column)

magnitude_feature_columns = list(
    dict.fromkeys(magnitude_feature_columns)
)

assert "symbol" in magnitude_feature_columns
assert "actual_return_pct" not in magnitude_feature_columns
assert "actual_magnitude_pct" not in magnitude_feature_columns
assert "trailing_magnitude_20d" in magnitude_feature_columns

print("Magnitude feature count:", len(magnitude_feature_columns))

Magnitude feature count: 38


In [7]:
final_fit_df = model_df[
    model_df["split"].isin(
        ["train", "valid"]
    )
].copy()

scored_df = model_df[
    model_df["split"].isin(
        ["train", "valid", "test"]
    )
].copy()

print("Final fitting rows:", len(final_fit_df))
print("Scored rows:", len(scored_df))

print("\nScored rows by split:")
print(scored_df["split"].value_counts())

print(
    "\nEmbargo rows excluded:",
    model_df["split"].eq("embargo").sum(),
)

Final fitting rows: 240564
Scored rows: 299220

Scored rows by split:
split
train    186555
test      58656
valid     54009
Name: count, dtype: int64

Embargo rows excluded: 2055


In [8]:
X_direction_fit = final_fit_df[
    direction_feature_columns
].copy()

y_direction_fit = (
    final_fit_df["actual_direction"]
    .eq(1)
    .astype("int8")
)

for column in direction_feature_columns:
    if column == "symbol":
        X_direction_fit[column] = (
            X_direction_fit[column]
            .astype("category")
        )

final_direction_model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=1000,
    num_leaves=31,
    learning_rate=0.05,
    min_child_samples=100,
    feature_fraction=0.60,
    bagging_fraction=0.80,
    bagging_freq=1,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

final_direction_model.fit(
    X_direction_fit,
    y_direction_fit,
    categorical_feature=["symbol"],
)

print("Final direction model trained.")

Final direction model trained.


In [10]:
model_df["log_actual_magnitude"] = np.log1p(
    model_df["actual_magnitude_pct"]
)

final_fit_df["log_actual_magnitude"] = np.log1p(
    final_fit_df["actual_magnitude_pct"]
)

X_magnitude_fit = final_fit_df[
    magnitude_feature_columns
].copy()

y_magnitude_fit = final_fit_df[
    "log_actual_magnitude"
].copy()

X_magnitude_fit["symbol"] = (
    X_magnitude_fit["symbol"]
    .astype("category")
)

selected_magnitude_iterations = int(
    magnitude_config["best_iteration"]
)

final_magnitude_model = lgb.LGBMRegressor(
    objective="regression_l1",
    n_estimators=selected_magnitude_iterations,
    num_leaves=31,
    learning_rate=0.03,
    min_child_samples=100,
    feature_fraction=0.70,
    bagging_fraction=0.80,
    bagging_freq=1,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

final_magnitude_model.fit(
    X_magnitude_fit,
    y_magnitude_fit,
    categorical_feature=["symbol"],
)

print("Final magnitude model trained.")
print(
    "Magnitude iterations:",
    selected_magnitude_iterations,
)

Final magnitude model trained.
Magnitude iterations: 84


In [11]:
X_direction_scored = scored_df[
    direction_feature_columns
].copy()

X_magnitude_scored = scored_df[
    magnitude_feature_columns
].copy()

X_direction_scored["symbol"] = (
    X_direction_scored["symbol"]
    .astype("category")
)

X_magnitude_scored["symbol"] = (
    X_magnitude_scored["symbol"]
    .astype("category")
)

scored_probability_up = (
    final_direction_model.predict_proba(
        X_direction_scored
    )[:, 1]
)

scored_pred_direction = np.where(
    scored_probability_up
    >= FINAL_DIRECTION_THRESHOLD,
    1,
    -1,
).astype("int8")

scored_pred_magnitude_log = (
    final_magnitude_model.predict(
        X_magnitude_scored
    )
)

scored_pred_magnitude_pct = np.clip(
    np.expm1(scored_pred_magnitude_log),
    0,
    None,
)

scored_df["raw_probability_up"] = (
    scored_probability_up
)

scored_df["pred_direction"] = (
    scored_pred_direction
)

scored_df["pred_magnitude_pct"] = (
    scored_pred_magnitude_pct
)

print(
    scored_df[
        [
            "raw_probability_up",
            "pred_direction",
            "pred_magnitude_pct",
        ]
    ].describe().T
)

                       count      mean       std       min       25%  \
raw_probability_up  299220.0  0.711768  0.199385  0.004412  0.622892   
pred_direction      299220.0  0.463037  0.886340 -1.000000 -1.000000   
pred_magnitude_pct  299220.0  0.458932  0.199888  0.187981  0.342800   

                         50%       75%       max  
raw_probability_up  0.774718  0.857094  0.996466  
pred_direction      1.000000  1.000000  1.000000  
pred_magnitude_pct  0.427968  0.530167  4.998508  


In [12]:
original_train_df = model_df[
    model_df["split"] == "train"
].copy()

original_valid_df = model_df[
    model_df["split"] == "valid"
].copy()

X_direction_train_only = original_train_df[
    direction_feature_columns
].copy()

X_direction_valid_only = original_valid_df[
    direction_feature_columns
].copy()

X_magnitude_train_only = original_train_df[
    magnitude_feature_columns
].copy()

X_magnitude_valid_only = original_valid_df[
    magnitude_feature_columns
].copy()

for frame in [
    X_direction_train_only,
    X_direction_valid_only,
    X_magnitude_train_only,
    X_magnitude_valid_only,
]:
    frame["symbol"] = frame[
        "symbol"
    ].astype("category")

y_direction_train_only = (
    original_train_df["actual_direction"]
    .eq(1)
    .astype("int8")
)

y_magnitude_train_only = np.log1p(
    original_train_df[
        "actual_magnitude_pct"
    ]
)

print("Original train rows:", len(original_train_df))
print("Original validation rows:", len(original_valid_df))

Original train rows: 186555
Original validation rows: 54009


In [13]:
shadow_direction_model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=1000,
    num_leaves=31,
    learning_rate=0.05,
    min_child_samples=100,
    feature_fraction=0.60,
    bagging_fraction=0.80,
    bagging_freq=1,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

shadow_direction_model.fit(
    X_direction_train_only,
    y_direction_train_only,
    categorical_feature=["symbol"],
)

validation_probability_up = (
    shadow_direction_model.predict_proba(
        X_direction_valid_only
    )[:, 1]
)

validation_pred_direction = np.where(
    validation_probability_up
    >= FINAL_DIRECTION_THRESHOLD,
    1,
    -1,
).astype("int8")

validation_direction_correct = (
    validation_pred_direction
    == original_valid_df[
        "actual_direction"
    ].to_numpy()
).astype("int8")

print(
    "Validation direction accuracy:",
    validation_direction_correct.mean(),
)

Validation direction accuracy: 0.6450406413745857


In [14]:
shadow_magnitude_model = lgb.LGBMRegressor(
    objective="regression_l1",
    n_estimators=selected_magnitude_iterations,
    num_leaves=31,
    learning_rate=0.03,
    min_child_samples=100,
    feature_fraction=0.70,
    bagging_fraction=0.80,
    bagging_freq=1,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

shadow_magnitude_model.fit(
    X_magnitude_train_only,
    y_magnitude_train_only,
    categorical_feature=["symbol"],
)

validation_magnitude_log = (
    shadow_magnitude_model.predict(
        X_magnitude_valid_only
    )
)

validation_pred_magnitude_pct = np.clip(
    np.expm1(validation_magnitude_log),
    0,
    None,
)

validation_absolute_magnitude_error = np.abs(
    validation_pred_magnitude_pct
    - original_valid_df[
        "actual_magnitude_pct"
    ].to_numpy()
)

print(
    pd.Series(
        validation_absolute_magnitude_error,
        name="validation_absolute_magnitude_error",
    ).describe()
)

count    54009.000000
mean         0.465267
std          0.799334
min          0.000012
25%          0.148090
50%          0.297335
75%          0.475042
max         33.881788
Name: validation_absolute_magnitude_error, dtype: float64


In [15]:
direction_confidence_training_df = (
    original_valid_df[
        [
            "symbol",
            "pred_date",
            "actual_direction",
            "actual_return_pct",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

direction_confidence_training_df[
    "raw_probability_up"
] = validation_probability_up

direction_confidence_training_df[
    "pred_direction"
] = validation_pred_direction

direction_confidence_training_df[
    "direction_correct"
] = validation_direction_correct

direction_confidence_training_df[
    "probability_margin"
] = np.abs(
    direction_confidence_training_df[
        "raw_probability_up"
    ]
    - FINAL_DIRECTION_THRESHOLD
)

direction_confidence_training_df[
    "probability_distance_from_half"
] = np.abs(
    direction_confidence_training_df[
        "raw_probability_up"
    ]
    - 0.50
)

direction_confidence_training_df[
    "emitted_probability"
] = np.where(
    direction_confidence_training_df[
        "pred_direction"
    ] == 1,
    direction_confidence_training_df[
        "raw_probability_up"
    ],
    1
    - direction_confidence_training_df[
        "raw_probability_up"
    ],
)

CONFIDENCE_BASE_FEATURES = [
    "raw_probability_up",
    "probability_margin",
    "probability_distance_from_half",
    "emitted_probability",
    "pred_direction",
]

direction_context_candidates = [
    "market_breadth",
    "aggregate_universe_volatility",
    "cross_sectional_return_dispersion",
    "day_of_week",
    "calendar_gap_days",
    "return_1d",
    "return_5d",
    "return_20d",
    "daily_volatility_20d",
    "overnight_volatility_20d",
    "volume_zscore_20d",
]

direction_context_features = [
    column
    for column in direction_context_candidates
    if column in original_valid_df.columns
]

for column in direction_context_features:
    direction_confidence_training_df[column] = (
        original_valid_df[column]
        .to_numpy()
    )

DIRECTION_CONFIDENCE_FEATURES = (
    CONFIDENCE_BASE_FEATURES
    + direction_context_features
)

print(
    "Direction-confidence feature count:",
    len(DIRECTION_CONFIDENCE_FEATURES),
)

Direction-confidence feature count: 15


In [16]:
X_direction_confidence_fit = (
    direction_confidence_training_df[
        DIRECTION_CONFIDENCE_FEATURES
    ]
    .copy()
)

y_direction_confidence_fit = (
    direction_confidence_training_df[
        "direction_correct"
    ]
    .copy()
)

direction_confidence_fill_values = (
    X_direction_confidence_fit
    .median(numeric_only=True)
)

X_direction_confidence_fit = (
    X_direction_confidence_fit
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
    .fillna(
        direction_confidence_fill_values
    )
)

final_direction_confidence_model = (
    LogisticRegression(
        C=0.25,
        max_iter=2000,
        class_weight=None,
        random_state=SEED,
    )
)

final_direction_confidence_model.fit(
    X_direction_confidence_fit,
    y_direction_confidence_fit,
)

print(
    "Final direction-confidence model trained."
)

Final direction-confidence model trained.


In [17]:
direction_confidence_scored_df = (
    scored_df[
        [
            "symbol",
            "pred_date",
            "raw_probability_up",
            "pred_direction",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

direction_confidence_scored_df[
    "probability_margin"
] = np.abs(
    direction_confidence_scored_df[
        "raw_probability_up"
    ]
    - FINAL_DIRECTION_THRESHOLD
)

direction_confidence_scored_df[
    "probability_distance_from_half"
] = np.abs(
    direction_confidence_scored_df[
        "raw_probability_up"
    ]
    - 0.50
)

direction_confidence_scored_df[
    "emitted_probability"
] = np.where(
    direction_confidence_scored_df[
        "pred_direction"
    ] == 1,
    direction_confidence_scored_df[
        "raw_probability_up"
    ],
    1
    - direction_confidence_scored_df[
        "raw_probability_up"
    ],
)

for column in direction_context_features:
    direction_confidence_scored_df[column] = (
        scored_df[column].to_numpy()
    )

X_direction_confidence_scored = (
    direction_confidence_scored_df[
        DIRECTION_CONFIDENCE_FEATURES
    ]
    .copy()
)

X_direction_confidence_scored = (
    X_direction_confidence_scored
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
    .fillna(
        direction_confidence_fill_values
    )
)

scored_conf_direction = (
    final_direction_confidence_model
    .predict_proba(
        X_direction_confidence_scored
    )[:, 1]
)

scored_conf_direction = np.clip(
    scored_conf_direction,
    0.500001,
    0.999999,
)

scored_df["conf_direction"] = (
    scored_conf_direction
)

pd.Series(
    scored_conf_direction,
    name="conf_direction",
).describe()

count    299220.000000
mean          0.671052
std           0.113021
min           0.500001
25%           0.542649
50%           0.703466
75%           0.761483
max           0.999512
Name: conf_direction, dtype: float64

In [18]:
magnitude_context_candidates = [
    "return_1d",
    "return_5d",
    "return_20d",
    "daily_volatility_20d",
    "overnight_volatility_20d",
    "short_term_volatility_5d",
    "gap_history_mean_20d",
    "gap_history_std_20d",
    "volume_zscore_20d",
    "volume_trend_5d_20d",
    "market_breadth",
    "aggregate_universe_volatility",
    "cross_sectional_return_dispersion",
    "day_of_week",
    "calendar_gap_days",
]

magnitude_context_features = [
    column
    for column in magnitude_context_candidates
    if column in original_valid_df.columns
]

print(
    "Magnitude-confidence context features:",
    magnitude_context_features,
)

Magnitude-confidence context features: ['return_1d', 'return_5d', 'return_20d', 'daily_volatility_20d', 'volume_zscore_20d', 'volume_trend_5d_20d', 'market_breadth', 'aggregate_universe_volatility', 'cross_sectional_return_dispersion', 'day_of_week', 'calendar_gap_days']


In [19]:
magnitude_confidence_training_df = (
    original_valid_df[
        [
            "symbol",
            "pred_date",
            "actual_magnitude_pct",
            "trailing_magnitude_20d",
            *magnitude_context_features,
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

magnitude_confidence_training_df[
    "pred_magnitude_pct"
] = validation_pred_magnitude_pct

magnitude_confidence_training_df[
    "absolute_magnitude_error"
] = (
    validation_absolute_magnitude_error
)

magnitude_confidence_training_df[
    "log_absolute_magnitude_error"
] = np.log1p(
    magnitude_confidence_training_df[
        "absolute_magnitude_error"
    ]
)

magnitude_confidence_training_df[
    "prediction_vs_baseline_ratio"
] = (
    magnitude_confidence_training_df[
        "pred_magnitude_pct"
    ]
    / magnitude_confidence_training_df[
        "trailing_magnitude_20d"
    ].replace(0, np.nan)
)

magnitude_confidence_training_df[
    "prediction_baseline_difference"
] = (
    magnitude_confidence_training_df[
        "pred_magnitude_pct"
    ]
    - magnitude_confidence_training_df[
        "trailing_magnitude_20d"
    ]
)

magnitude_confidence_training_df[
    "log_predicted_magnitude"
] = np.log1p(
    magnitude_confidence_training_df[
        "pred_magnitude_pct"
    ].clip(lower=0)
)

MAGNITUDE_CONFIDENCE_FEATURES = [
    "pred_magnitude_pct",
    "trailing_magnitude_20d",
    "log_predicted_magnitude",
    "prediction_vs_baseline_ratio",
    "prediction_baseline_difference",
    *magnitude_context_features,
]

MAGNITUDE_CONFIDENCE_FEATURES = list(
    dict.fromkeys(
        MAGNITUDE_CONFIDENCE_FEATURES
    )
)

print(
    "Magnitude-confidence feature count:",
    len(MAGNITUDE_CONFIDENCE_FEATURES),
)

Magnitude-confidence feature count: 16


In [20]:
X_magnitude_confidence_fit = (
    magnitude_confidence_training_df[
        MAGNITUDE_CONFIDENCE_FEATURES
    ]
    .copy()
)

y_magnitude_confidence_fit = (
    magnitude_confidence_training_df[
        "log_absolute_magnitude_error"
    ]
    .copy()
)

magnitude_confidence_fill_values = (
    X_magnitude_confidence_fit
    .median(numeric_only=True)
)

X_magnitude_confidence_fit = (
    X_magnitude_confidence_fit
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
    .fillna(
        magnitude_confidence_fill_values
    )
)

final_magnitude_confidence_model = (
    lgb.LGBMRegressor(
        objective="regression_l1",
        n_estimators=750,
        num_leaves=15,
        learning_rate=0.03,
        min_child_samples=150,
        feature_fraction=0.80,
        bagging_fraction=0.80,
        bagging_freq=1,
        reg_lambda=2.0,
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1,
    )
)

final_magnitude_confidence_model.fit(
    X_magnitude_confidence_fit,
    y_magnitude_confidence_fit,
)

training_expected_error = np.clip(
    np.expm1(
        final_magnitude_confidence_model.predict(
            X_magnitude_confidence_fit
        )
    ),
    0,
    None,
)

sorted_training_expected_error = np.sort(
    training_expected_error
)

print(
    "Final magnitude-confidence model trained."
)

print(
    pd.Series(
        training_expected_error,
        name="training_expected_error",
    ).describe()
)

Final magnitude-confidence model trained.
count    54009.000000
mean         0.372476
std          0.415004
min          0.124805
25%          0.253573
50%          0.300091
75%          0.368059
max          8.380573
Name: training_expected_error, dtype: float64


In [21]:
magnitude_confidence_scored_df = (
    scored_df[
        [
            "symbol",
            "pred_date",
            "pred_magnitude_pct",
            "trailing_magnitude_20d",
            *magnitude_context_features,
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

magnitude_confidence_scored_df[
    "prediction_vs_baseline_ratio"
] = (
    magnitude_confidence_scored_df[
        "pred_magnitude_pct"
    ]
    / magnitude_confidence_scored_df[
        "trailing_magnitude_20d"
    ].replace(0, np.nan)
)

magnitude_confidence_scored_df[
    "prediction_baseline_difference"
] = (
    magnitude_confidence_scored_df[
        "pred_magnitude_pct"
    ]
    - magnitude_confidence_scored_df[
        "trailing_magnitude_20d"
    ]
)

magnitude_confidence_scored_df[
    "log_predicted_magnitude"
] = np.log1p(
    magnitude_confidence_scored_df[
        "pred_magnitude_pct"
    ].clip(lower=0)
)

X_magnitude_confidence_scored = (
    magnitude_confidence_scored_df[
        MAGNITUDE_CONFIDENCE_FEATURES
    ]
    .copy()
)

X_magnitude_confidence_scored = (
    X_magnitude_confidence_scored
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
    .fillna(
        magnitude_confidence_fill_values
    )
)

scored_expected_magnitude_error = np.clip(
    np.expm1(
        final_magnitude_confidence_model.predict(
            X_magnitude_confidence_scored
        )
    ),
    0,
    None,
)

error_percentile = np.searchsorted(
    sorted_training_expected_error,
    scored_expected_magnitude_error,
    side="right",
) / len(
    sorted_training_expected_error
)

scored_conf_magnitude = np.clip(
    1.0 - error_percentile,
    0.0,
    1.0,
)

scored_df["expected_magnitude_error"] = (
    scored_expected_magnitude_error
)

scored_df["conf_magnitude"] = (
    scored_conf_magnitude
)

pd.Series(
    scored_conf_magnitude,
    name="conf_magnitude",
).describe()

count    299220.000000
mean          0.534228
std           0.298767
min           0.000093
25%           0.265826
50%           0.546964
75%           0.803088
max           1.000000
Name: conf_magnitude, dtype: float64

In [22]:
required_prediction_outputs = [
    "pred_direction",
    "conf_direction",
    "pred_magnitude_pct",
    "conf_magnitude",
]

assert scored_df[
    required_prediction_outputs
].notna().all().all()

assert scored_df[
    "pred_direction"
].isin([-1, 1]).all()

assert scored_df[
    "conf_direction"
].between(
    0.5,
    1.0,
).all()

assert scored_df[
    "pred_magnitude_pct"
].ge(0).all()

assert scored_df[
    "conf_magnitude"
].between(
    0.0,
    1.0,
).all()

assert not scored_df.duplicated(
    ["symbol", "pred_date"]
).any()

print("All four final outputs passed validation.")

print("\nRows by split:")
print(
    scored_df["split"]
    .value_counts()
)

print("\nPrediction summary:")
print(
    scored_df[
        required_prediction_outputs
    ].describe().T
)

All four final outputs passed validation.

Rows by split:
split
train    186555
test      58656
valid     54009
Name: count, dtype: int64

Prediction summary:
                       count      mean       std       min       25%  \
pred_direction      299220.0  0.463037  0.886340 -1.000000 -1.000000   
conf_direction      299220.0  0.671052  0.113021  0.500001  0.542649   
pred_magnitude_pct  299220.0  0.458932  0.199888  0.187981  0.342800   
conf_magnitude      299220.0  0.534228  0.298767  0.000093  0.265826   

                         50%       75%       max  
pred_direction      1.000000  1.000000  1.000000  
conf_direction      0.703466  0.761483  0.999512  
pred_magnitude_pct  0.427968  0.530167  4.998508  
conf_magnitude      0.546964  0.803088  1.000000  
